# RSNA Knee — Submission v2: ensemble 5-fold model kita sendiri (bukan komunitas)

**Konteks**: submission pertama yang benar-benar pakai model kita sendiri, bukan reuse kode
komunitas (`kaggle_kernel_submission/` v1 pakai 20-model pilkwang). Sumber: 5 checkpoint dari
`rsna-knee-modeling-v18-kfold-5-tpu` (k-fold CV, OOF macro AUC **0,8068** -- metrik paling
terpercaya proyek ini sejauh ini, lihat `PROGRESS.md` §28).

**Arsitektur**: identik dgn v15/v18 (DINOv2-base, unfreeze 6 blok terakhir -- tidak relevan saat
inferensi, cuma perlu struktur cocok utk `load_state_dict`, slot attention 6-slot, K_SLICE=4
subsample linspace dari slice fisik-terurut per series).

**Beda dari preprocessing/modeling kernel lain**: TIDAK ada cache `.npz` utk `test_series` (cache
preprocessing v1/v2/v3 semuanya cuma proses `train_series`) -- notebook ini decode DICOM MENTAH
dari `test_series` langsung saat inferensi, pakai fungsi preprocessing YANG SAMA (`get_sorted_files`
urutan fisik, `preprocess_slice` clip per-grup) disalin dari `kaggle_kernel_preprocessing_v2`, tapi
dipakai on-the-fly (bukan di-cache dulu) krn test set cuma dilihat sekali saat submission sungguhan.
Statistik normalisasi per-grup (`group_norm_stats.csv`) DIPINJAM dari preprocessing v2 (dihitung
dari sampel TRAIN, dipakai konsisten utk TEST -- standar praktik, bukan dihitung ulang dari test).

**Ensemble**: 5 model dijalankan independen per studi, prediksi sigmoid mentah DIRATA-RATAKAN
(mean probability, bukan rank-blend -- kelima model arsitektur & skala sama, beda cuma subset
training, rata-rata sederhana sudah tepat). Hasil akhir ditulis dlm bentuk **percentile rank**
per label (pola sama dgn kode komunitas pilkwang) krn metrik AUC cuma peduli urutan.

**Status saat ini**: `enable_gpu=false, enable_tpu=false` (CPU) -- versi UJI COBA dulu di `test_series`
stub (3 studi), murah & cepat terlepas dari akselerator. Begitu tervalidasi jalan bersih, baru
diputuskan apakah benar-benar disubmit (perlu koordinasi tim, kuota submission 5x/hari
UNTUK SELURUH TIM -- lihat `MEMORY.md` §4) & dgn GPU T4 (pola submission-ready standar proyek ini,
`enable_internet=false`) setelah kuota GPU refresh (~22 Agustus).


## 0. Setup

In [ ]:
import os, sys, json, time, hashlib, traceback, warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from scipy import ndimage as ndi

warnings.filterwarnings("ignore")
T0 = time.time()
def tlog(msg):
    print(f"[t={time.time()-T0:7.1f}s] {msg}", flush=True)

def ensure(pkg, import_name=None):
    import importlib
    name = import_name or pkg
    try:
        importlib.import_module(name); return True
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q {pkg}")
        try:
            importlib.import_module(name); return True
        except Exception as e:
            print(f"[warn] gagal import/install {pkg}: {e}"); return False

HAVE_PYDICOM = ensure("pydicom")
ensure("transformers")
ensure("scikit-learn", "sklearn")
import pydicom
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# Deteksi device: XLA (TPU) -> CUDA -> CPU. Sama pola fallback aman spt kernel modeling lain --
# submission SUNGGUHAN kompetisi ini historis pakai GPU T4 (lihat kaggle_kernel_submission/ v1),
# jadi versi ini sengaja CPU-only (enable_gpu=false di kernel-metadata) utk uji coba murah dulu.
XLA_OK = False
xm = None
try:
    import torch_xla.core.xla_model as xm
    _xla_dev = xm.xla_device()
    _ = torch.zeros(1, device=_xla_dev) + 1
    XLA_OK = True
except Exception:
    pass

if XLA_OK:
    DEVICE = _xla_dev
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
tlog(f"device: {DEVICE} | xla: {XLA_OK}")


## 1. Cari sumber data (kompetisi, cache preprocessing v2, 5 checkpoint v18, DINOv2)

In [ ]:
kaggle_input = Path("/kaggle/input")
CANDIDATE_ROOTS = [Path("/kaggle/input/rsna-knee-abnormality-detection"),
                    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")]
if kaggle_input.exists():
    for pattern in ("*/test.csv", "*/*/test.csv"):
        for hit in kaggle_input.glob(pattern):
            CANDIDATE_ROOTS.insert(0, hit.parent)
DATA_DIR = next((p for p in CANDIDATE_ROOTS if p.exists() and (p / "test.csv").exists()), None)
assert DATA_DIR is not None, "competition data (test.csv) tidak ketemu -- cek competition_sources"
tlog(f"DATA_DIR: {DATA_DIR}")

test_df = pd.read_csv(DATA_DIR / "test.csv")
test_series = pd.read_csv(DATA_DIR / "test_series.csv")
TEST_DIR = DATA_DIR / "test_series"
tlog(f"test_df: {test_df.shape} | test_series: {test_series.shape}")

def find_dir_containing(filename, hint=""):
    if not kaggle_input.exists():
        return None
    candidates = []
    for root, dirs, files in os.walk(kaggle_input):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series", "image_cache")]
        if filename in files:
            candidates.append(Path(root))
    if not candidates:
        return None
    if hint:
        for c in candidates:
            if hint.lower() in str(c).lower():
                return c
    return candidates[0]

PREP_DIR = find_dir_containing("group_norm_stats.csv", "preprocessing")
tlog(f"PREP_DIR (sumber group_norm_stats.csv): {PREP_DIR}")
assert PREP_DIR is not None, "preprocessing v2 kernel belum di-attach (kernel_sources) -- butuh group_norm_stats.csv"

CKPT_PATHS = sorted(kaggle_input.rglob("best_model_fold*.pt"))
tlog(f"Checkpoint fold ketemu: {len(CKPT_PATHS)}")
for p in CKPT_PATHS:
    tlog(f"  {p}")
assert len(CKPT_PATHS) > 0, "checkpoint fold (best_model_fold*.pt) tidak ketemu -- cek kernel_sources v18"

def find_dinov2(variant="base"):
    hits = []
    for root, dirs, files in os.walk(kaggle_input):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series", "image_cache")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None

DINOV2_DIR = find_dinov2("base")
tlog(f"DINOV2_DIR: {DINOV2_DIR}")
assert DINOV2_DIR is not None, "DINOv2 model belum di-attach (model_sources)"


## 2. Preprocessing on-the-fly (disalin dari `kaggle_kernel_preprocessing_v2`)

`test_series` tidak punya cache -- fungsi di bawah SAMA PERSIS dgn preprocessing v2 (urutan fisik
`ImagePositionPatient` + clip per-grup + resample fisik + resize), dipakai langsung saat inferensi.
`GROUP_STATS` dipinjam dari `group_norm_stats.csv` (dihitung preprocessing v2 dari sampel TRAIN).


In [ ]:
group_norm_stats = pd.read_csv(PREP_DIR / "group_norm_stats.csv")
GROUP_STATS = {r["group"]: (r["p1"], r["p99"]) for _, r in group_norm_stats.iterrows()}
FALLBACK_STATS = (float(group_norm_stats["p1"].median()), float(group_norm_stats["p99"].median()))
tlog(f"GROUP_STATS: {len(GROUP_STATS)} grup dimuat dari preprocessing v2")

IMG_SIZE = 224
TARGET_MM_PER_PX = 0.5
K_SLICE = 4  # v15/v18: subsample dari urutan fisik, arsitektur checkpoint mengharap input ini

def list_dcm(study, series):
    d = TEST_DIR / study / series
    return sorted(d.glob("*.dcm")) if d.exists() else []

def get_sorted_files(files):
    """Sama persis dgn preprocessing v2 -- urutkan file berdasarkan posisi fisik (proyeksi
    ImagePositionPatient ke sumbu normal slice), fallback SliceLocation -> InstanceNumber ->
    urutan file asli."""
    keyed = []
    for i, f in enumerate(files):
        pos = None
        try:
            ds = pydicom.dcmread(f, force=True, stop_before_pixels=True)
            try:
                iop = [float(x) for x in ds.ImageOrientationPatient]
                row_dir, col_dir = np.array(iop[:3]), np.array(iop[3:])
                normal = np.cross(row_dir, col_dir)
                ipp = np.array([float(x) for x in ds.ImagePositionPatient])
                pos = float(np.dot(normal, ipp))
            except Exception:
                pass
            if pos is None:
                try:
                    pos = float(ds.SliceLocation)
                except Exception:
                    pass
            if pos is None:
                try:
                    pos = float(ds.InstanceNumber)
                except Exception:
                    pass
        except Exception:
            pass
        if pos is None:
            pos = float(i)
        keyed.append((pos, i, f))
    keyed.sort(key=lambda t: (t[0], t[1]))
    return [f for _, _, f in keyed]

def preprocess_slice(ds, group_key):
    """Sama persis dgn preprocessing v2/v1: RescaleSlope/Intercept -> invert MONOCHROME1 ->
    clip per-grup (GROUP_STATS) -> resample fisik (PixelSpacing) -> resize/crop ke 224x224."""
    arr = ds.pixel_array.astype(np.float64)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept

    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        arr = arr.max() - arr

    p1, p99 = GROUP_STATS.get(group_key, FALLBACK_STATS)
    arr = np.clip(arr, p1, p99)
    rng = (p99 - p1) if (p99 - p1) > 1e-6 else 1.0
    arr = (arr - p1) / rng

    ps = getattr(ds, "PixelSpacing", None)
    mm_per_px = float(ps[0]) if ps else TARGET_MM_PER_PX
    zoom_factor = mm_per_px / TARGET_MM_PER_PX
    if abs(zoom_factor - 1.0) > 0.02:
        arr = ndi.zoom(arr, zoom_factor, order=1)

    h, w = arr.shape[:2]
    if h != IMG_SIZE or w != IMG_SIZE:
        pad_h, pad_w = max(0, IMG_SIZE - h), max(0, IMG_SIZE - w)
        if pad_h or pad_w:
            arr = np.pad(arr, ((pad_h // 2, pad_h - pad_h // 2), (pad_w // 2, pad_w - pad_w // 2)), mode="constant")
        h, w = arr.shape[:2]
        top, left = max(0, (h - IMG_SIZE) // 2), max(0, (w - IMG_SIZE) // 2)
        arr = arr[top: top + IMG_SIZE, left: left + IMG_SIZE]
        if arr.shape != (IMG_SIZE, IMG_SIZE):
            arr = np.pad(arr, ((0, IMG_SIZE - arr.shape[0]), (0, IMG_SIZE - arr.shape[1])), mode="constant")

    return np.clip(arr * 255, 0, 255).astype(np.uint8)

def decode_series_kslice(study, series):
    """Decode LANGSUNG K_SLICE=4 slice dari urutan fisik (bukan cache 16 -> subsample lagi spt
    training -- di sini cukup decode yg dibutuhkan saja, hemat kompute krn cuma sekali pakai)."""
    files = list_dcm(study, series)
    if not files:
        return None
    files_sorted = get_sorted_files(files)
    idxs = np.linspace(0, len(files_sorted) - 1, min(K_SLICE, len(files_sorted))).astype(int)
    idxs = sorted(set(idxs.tolist()))
    slices = []
    for fi in idxs:
        try:
            ds = pydicom.dcmread(files_sorted[fi], force=True)
            slices.append(ds)
        except Exception:
            continue
    return slices


## 3. Skema slot + arsitektur model (disalin dari `kaggle_kernel_modeling_kfold`)

In [ ]:
LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker\'s","Contusion","Fracture"]
LABEL_KEY = [c.replace("\'", "") for c in LABEL_COLS]

def slot_index(plane, fs, fatsup):
    plane = str(plane)
    is_fluid = (int(fs) == 1) if pd.notna(fs) else False
    base = {"Sagittal": 0, "Coronal": 1, "Axial": 2}.get(plane, 0)
    return base + (3 if is_fluid else 0)

N_SLOT = 6
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (0, 1), "Lateral OA": (0, 1), "PF OA": (2, 5),
    "Effusion": (3, 4, 5), "Synovitis": (3, 4, 5),
    "Bakers": (2, 3, 5), "Contusion": (3, 4, 5), "Fracture": (0, 1, 2),
}
SLOT_PRIOR_STRENGTH = 0.55
SLOT_PRIOR_IDX = {k.replace("\'", ""): v for k, v in SLOT_PRIOR_TABLE.items()}


class SlotHead(nn.Module):
    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        prior = torch.zeros(n_out, n_slot)
        for i, target in enumerate(LABEL_KEY):
            for s in SLOT_PRIOR_IDX.get(target, ()):
                prior[i, s] = SLOT_PRIOR_STRENGTH
        self.register_buffer("slot_prior", prior)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


class KneeModel(nn.Module):
    """Struktur SAMA PERSIS dgn v15/v18 (perlu cocok utk load_state_dict) -- unfreeze_last tidak
    relevan lagi saat inferensi murni (eval + no_grad), tapi dipertahankan biar konsisten."""
    def __init__(self, backbone_path, unfreeze_last=6):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(str(backbone_path))
        n_layer = len(self.backbone.encoder.layer)
        dim = self.backbone.config.hidden_size
        self.head = SlotHead(dim * 2, N_SLOT, len(LABEL_COLS))
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask):
        B, S, K, H, W = imgs.shape
        x = imgs.reshape(B * S * K, 1, H, W).div(255.0)
        x = x.repeat(1, 3, 1, 1)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        cls = out[:, 0]
        patch_mean = out[:, 1:].mean(1)
        feat = torch.cat([cls, patch_mean], dim=1)
        feat = feat.reshape(B, S, K, -1).mean(2)
        return self.head(feat, mask)


## 4. Muat 5 model fold (ensemble)

In [ ]:
models = []
for p in CKPT_PATHS:
    m = KneeModel(DINOV2_DIR).to(DEVICE)
    state = torch.load(p, map_location="cpu")
    m.load_state_dict(state)
    m.eval()
    models.append(m)
    tlog(f"model dimuat: {p.name}")
tlog(f"Total {len(models)} model dalam ensemble.")


## 5. Inferensi per studi (decode on-the-fly -> 6 slot -> ensemble 5 model)

In [ ]:
def infer_study(study_id):
    """Return np.array(12,) prediksi rata-rata 5 model, atau None kalau studi tidak ada series."""
    rows = test_series[test_series["StudyInstanceUID"] == study_id]
    if rows.empty:
        return None

    imgs = np.zeros((N_SLOT, K_SLICE, IMG_SIZE, IMG_SIZE), dtype=np.float32)
    mask = np.zeros(N_SLOT, dtype=np.float32)
    slot_series = defaultdict(list)
    for _, r in rows.iterrows():
        slot = slot_index(r["Anatomical_Plane"], r["Fluid_Sensitive"], r["Fat_Suppression"])
        slot_series[slot].append(r["SeriesInstanceUID"])

    for slot, series_ids in slot_series.items():
        group_key = None
        slot_slices = []
        for sid in series_ids[:2]:
            row = rows[rows["SeriesInstanceUID"] == sid].iloc[0]
            group_key = f"{row['Anatomical_Plane']}|FS{row['Fluid_Sensitive']}|Fat{row['Fat_Suppression']}"
            dss = decode_series_kslice(study_id, sid)
            if not dss:
                continue
            arr8 = np.stack([preprocess_slice(ds, group_key) for ds in dss], axis=0)  # (n<=K_SLICE, H, W)
            if arr8.shape[0] < K_SLICE:  # series terlalu pendek -- ulang slice terakhir biar shape tetap
                pad = np.repeat(arr8[-1:], K_SLICE - arr8.shape[0], axis=0)
                arr8 = np.concatenate([arr8, pad], axis=0)
            slot_slices.append(arr8[:K_SLICE])
        if slot_slices:
            imgs[slot] = np.stack(slot_slices, axis=0).astype(np.float32).mean(axis=0)
            mask[slot] = 1.0

    if mask.sum() == 0:
        return None

    imgs_t = torch.from_numpy(imgs).unsqueeze(0).to(DEVICE)   # (1, N_SLOT, K_SLICE, H, W)
    mask_t = torch.from_numpy(mask).unsqueeze(0).to(DEVICE)   # (1, N_SLOT)
    preds = []
    with torch.no_grad():
        for m in models:
            logits = m(imgs_t, mask_t)
            preds.append(torch.sigmoid(logits).cpu().numpy()[0])
    return np.mean(preds, axis=0)   # (12,) -- rata-rata sederhana 5 model (arsitektur & skala sama)


# --- benchmark fallback dulu (0.5 semua) -- ada dari detik pertama, jaga2 kalau loop di bawah gagal ---
_bench = test_df[["StudyInstanceUID"]].copy()
for c in LABEL_COLS:
    _bench[c] = 0.5
_bench.to_csv("submission.csv", index=False)
tlog("submission.csv fallback (0.5 semua) ditulis.")

pred_rows, pred_studies = [], []
for sid in test_df["StudyInstanceUID"]:
    p = infer_study(sid)
    if p is not None:
        pred_rows.append(p)
        pred_studies.append(sid)
    else:
        tlog(f"[warn] studi {sid} tidak ada series/gagal decode -- fallback 0.5 tetap dipakai")
tlog(f"Inferensi selesai: {len(pred_studies)}/{len(test_df)} studi berhasil.")


## 6. Tulis submission.csv final (percentile rank per label, pola sama dgn kode komunitas)

In [ ]:
if pred_rows:
    pred_mat = np.stack(pred_rows, axis=0)
    sub = pd.DataFrame(pd.DataFrame(pred_mat).rank(pct=True).values, columns=LABEL_COLS)
    sub.insert(0, "StudyInstanceUID", pred_studies)
    sub = test_df[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
    sub[LABEL_COLS] = sub[LABEL_COLS].fillna(0.5)
    sub.to_csv("submission.csv", index=False)
    tlog(f"submission.csv final ditulis: {sub.shape}")
    print(sub.head())
else:
    tlog("[warn] tidak ada prediksi berhasil sama sekali -- submission.csv TETAP fallback 0.5.")


## Catatan

- **Ensemble 5 model fold v18** (OOF macro AUC 0,8068) -- rata-rata sigmoid sederhana, bukan
  rank-blend (kelima model arsitektur & skala identik, cuma beda subset training).
- Preprocessing test-time SAMA PERSIS dgn preprocessing v2 (urutan fisik + clip per-grup), TAPI
  on-the-fly (tidak di-cache) krn test set cuma dilihat sekali saat submission sungguhan.
- Prediksi ditulis sbg percentile rank per label (pola komunitas) -- metrik macro AUC cuma peduli
  urutan, rank membuat file ini juga aman kalau nanti mau di-rank-blend dgn sumber lain.
- **BELUM disubmit ke leaderboard sungguhan** -- versi ini CPU-only utk validasi logika dulu di
  `test_series` stub (3 studi). Sebelum submit sungguhan: (1) koordinasi dgn tim (kuota 5x/hari
  utk SELURUH TIM), (2) pindah ke GPU T4 + `enable_internet=false` (pola standar submission-ready
  proyek ini) setelah kuota GPU refresh (~22 Agustus).
